# Julia solutions for `S01.ipynb`

This notebook rewrites the exercises in **Julia**.

## Packages
If you do not have the packages yet, run this once in Julia:

```julia
using Pkg
Pkg.add(["Plots", "JuMP", "Ipopt", "HiGHS", "LinearAlgebra"])
```

In [ ]:
using JuMP
using Ipopt
using HiGHS
using LinearAlgebra


## 1) Plot `sin(x)` on `[-π/4, 3π/4]`

In [ ]:
x = range(-pi/4, 3pi/4, length=400)
y = sin.(x)

plot(
    x, y,
    xlabel = "x",
    ylabel = "sin(x)",
    label = "sin(x)",
    title = "sin(x) on [-π/4, 3π/4]",
    legend = :topright
)

## 2) Plot `x*sin(x)` on `[-10π, 10π]`

In [ ]:
x = range(-10pi, 10pi, length=2000)
y = x .* sin.(x)

plot(
    x, y,
    xlabel = "x",
    ylabel = "x·sin(x)",
    label = "x·sin(x)",
    title = "x·sin(x) on [-10π, 10π]",
    legend = :topright
)

## 3) Cylinder Problem

Using the standard nonlinear model:

- decision variables: radius `r` and height `h`
- objective:

`max  Nπr²h - c₁πr² - c₂(πr²h)(πr² + 2πrh)`

Data:
- `N = 10`
- `c₁ = 2`
- `c₂ = 0.5`

In [ ]:
N = 10.0
c1 = 2.0
c2 = 0.5

model_cyl = Model(Ipopt.Optimizer)
set_silent(model_cyl)

@variable(model_cyl, r >= 0.0)
@variable(model_cyl, h >= 0.0)

@NLexpression(model_cyl, V, pi * r^2 * h)
@NLobjective(
    model_cyl, Max,
    N * pi * r^2 * h - c1 * pi * r^2 - c2 * V * (pi * r^2 + 2pi * r * h)
)

set_start_value(r, 0.8)
set_start_value(h, 1.7)

optimize!(model_cyl)

r_opt = value(r)
h_opt = value(h)
V_opt = pi * r_opt^2 * h_opt
profit_opt = objective_value(model_cyl)

println("Cylinder problem")
println("r*      = ", r_opt)
println("h*      = ", h_opt)
println("V*      = ", V_opt)
println("profit* = ", profit_opt)

## 4) Awning Problem

Standard formulation:

- minimize `sqrt(x² + y²)`
- subject to `y - w*y/x >= h`
- `x >= 0`, `y >= 0`

Data:
- `h = 2`
- `w = 3`
- initial guess `(x, y) = (1.0, 1.0)`

In [ ]:
h = 2.0
w = 3.0

model_awning = Model(Ipopt.Optimizer)
set_silent(model_awning)

@variable(model_awning, x >= 1e-6)
@variable(model_awning, y >= 0.0)

@NLobjective(model_awning, Min, sqrt(x^2 + y^2))
@NLconstraint(model_awning, y - w * y / x >= h)

set_start_value(x, 5.0)
set_start_value(y, 4.0)

optimize!(model_awning)

x_opt = value(x)
y_opt = value(y)
z_opt = objective_value(model_awning)

println("Awning problem")
println("x* = ", x_opt)
println("y* = ", y_opt)
println("minimum awning length = ", z_opt)

## 5) Packing Problem

The original cell only says **"Solve the Packing Problem"** and does **not** provide data or a full formulation.

To keep the notebook usable, below I included a **generic 1D bin-packing model** in Julia.  
You can replace `item_sizes` and `bin_capacity` with the values from your class if needed.

In [ ]:
# Generic 1D bin-packing example
item_sizes = [4, 8, 1, 4, 2, 1, 10]
bin_capacity = 10

n_items = length(item_sizes)
max_bins = n_items

model_pack = Model(HiGHS.Optimizer)
set_silent(model_pack)

@variable(model_pack, x[1:n_items, 1:max_bins], Bin)   # item i assigned to bin j
@variable(model_pack, y[1:max_bins], Bin)             # bin j used

@objective(model_pack, Min, sum(y[j] for j in 1:max_bins))

@constraint(model_pack, [i in 1:n_items], sum(x[i, j] for j in 1:max_bins) == 1)
@constraint(model_pack, [j in 1:max_bins],
    sum(item_sizes[i] * x[i, j] for i in 1:n_items) <= bin_capacity * y[j]
)

optimize!(model_pack)

used_bins = Int(round(objective_value(model_pack)))
println("Packing problem (generic model)")
println("Minimum number of bins = ", used_bins)

for j in 1:max_bins
    if value(y[j]) > 0.5
        packed = [i for i in 1:n_items if value(x[i, j]) > 0.5]
        println("Bin ", j, ": items ", packed, " -> sizes ", item_sizes[packed],
                ", total = ", sum(item_sizes[packed]))
    end
end

## 6) 3-bus Optimal Power Flow (AC-OPF)

Data interpreted from the figure:

- generators at buses 1 and 2
- load at bus 3: `0.8 + j0.6`
- line admittance on all branches: `1 / (0.01 + j0.1)`
- thermal limits:
  - `Smax₁₂ = Smax₂₁ = 0.25`
  - `Smax₁₃ = Smax₃₁ = 2`
  - `Smax₂₃ = Smax₃₂ = 2`
- voltage bounds: `0.95 <= vᵢ <= 1.10`
- angle reference: `δ₃ = 0`
- generator limits from the figure
- linear generation cost: `2 p₁ + 1 p₂`

In [ ]:
using JuMP
using Ipopt

# =========================
# 3-bus Optimal Power Flow
# =========================

# Line data
z = 0.01 + 0.1im
y = 1 / z

# Bus admittance matrix
Y = [
    2y   -y   -y
    -y   2y   -y
    -y   -y   2y
]

G = real.(Y)
B = imag.(Y)

# Loads (only bus 3 has demand)
Pd = [0.0, 0.0, 0.8]
Qd = [0.0, 0.0, 0.6]

# Generation costs
c = [2.0, 1.0]

# Apparent power limits
Smax = Dict(
    (1, 2) => 0.25, (2, 1) => 0.25,
    (1, 3) => 2.0,  (3, 1) => 2.0,
    (2, 3) => 2.0,  (3, 2) => 2.0,
)

let
    model_opf = Model(Ipopt.Optimizer)
    set_silent(model_opf)

    # Variables
    @variable(model_opf, 0.95 <= v[1:3] <= 1.10)
    @variable(model_opf, δ[1:3])

    @variable(model_opf, 0.0 <= p1 <= 3.0)
    @variable(model_opf, -2.0 <= q1 <= 2.0)
    @variable(model_opf, 0.0 <= p2 <= 0.8)
    @variable(model_opf, -2.0 <= q2 <= 2.0)

    # Slack/reference bus
    fix(δ[3], 0.0; force=true)

    # Anonymous nonlinear expressions avoid name conflicts if the cell is rerun
    Pg = Vector{Any}(undef, 3)
    Qg = Vector{Any}(undef, 3)
    Pg[1] = @NLexpression(model_opf, p1)
    Qg[1] = @NLexpression(model_opf, q1)
    Pg[2] = @NLexpression(model_opf, p2)
    Qg[2] = @NLexpression(model_opf, q2)
    Pg[3] = @NLexpression(model_opf, 0.0)
    Qg[3] = @NLexpression(model_opf, 0.0)

    # Power balance at each bus
    for i in 1:3
        @NLconstraint(model_opf,
            Pg[i] - Pd[i] ==
            sum(v[i] * v[j] * (G[i, j] * cos(δ[i] - δ[j]) + B[i, j] * sin(δ[i] - δ[j])) for j in 1:3)
        )
        @NLconstraint(model_opf,
            Qg[i] - Qd[i] ==
            sum(v[i] * v[j] * (G[i, j] * sin(δ[i] - δ[j]) - B[i, j] * cos(δ[i] - δ[j])) for j in 1:3)
        )
    end

    # Branch flows
    P = Dict{Tuple{Int,Int},Any}()
    Q = Dict{Tuple{Int,Int},Any}()

    for (i, j) in [(1,2), (2,1), (1,3), (3,1), (2,3), (3,2)]
        gij = -real(Y[i, j])
        bij = -imag(Y[i, j])

        P[(i,j)] = @NLexpression(model_opf,
            v[i]^2 * gij - v[i] * v[j] * (gij * cos(δ[i] - δ[j]) + bij * sin(δ[i] - δ[j]))
        )
        Q[(i,j)] = @NLexpression(model_opf,
            -v[i]^2 * bij - v[i] * v[j] * (gij * sin(δ[i] - δ[j]) - bij * cos(δ[i] - δ[j]))
        )

        @NLconstraint(model_opf, P[(i,j)]^2 + Q[(i,j)]^2 <= Smax[(i,j)]^2)
    end

    # Objective
    @objective(model_opf, Min, c[1] * p1 + c[2] * p2)

    # Initial values
    set_start_value.(v, [1.0, 1.0, 1.0])
    set_start_value.(δ, [0.0, 0.0, 0.0])
    set_start_value(p1, 0.3)
    set_start_value(q1, 0.2)
    set_start_value(p2, 0.5)
    set_start_value(q2, 0.3)

    optimize!(model_opf)

    println("Status: ", termination_status(model_opf))
    println("Valor ótimo = ", objective_value(model_opf))
    println()

    println("p1 = ", value(p1))
    println("q1 = ", value(q1))
    println("p2 = ", value(p2))
    println("q2 = ", value(q2))
    println()

    println("v1 = ", value(v[1]))
    println("v2 = ", value(v[2]))
    println("v3 = ", value(v[3]))
    println()

    println("d1 = ", value(δ[1]))
    println("d2 = ", value(δ[2]))
    println("d3 = ", value(δ[3]))
    println()

    println("Fluxos:")
    println("P12 = ", value(P[(1,2)]), "   Q12 = ", value(Q[(1,2)]))
    println("P21 = ", value(P[(2,1)]), "   Q21 = ", value(Q[(2,1)]))
    println("P13 = ", value(P[(1,3)]), "   Q13 = ", value(Q[(1,3)]))
    println("P31 = ", value(P[(3,1)]), "   Q31 = ", value(Q[(3,1)]))
    println("P23 = ", value(P[(2,3)]), "   Q23 = ", value(Q[(2,3)]))
    println("P32 = ", value(P[(3,2)]), "   Q32 = ", value(Q[(3,2)]))
end

## 7) Linear Regression with 3 variables

Model:
`y = β₀ + β₁x₁ + β₂x₂ + β₃x₃`

Solved by least squares.

In [ ]:
X = [
    1.0  1.0  0.5  1.2
    1.0  2.0  1.0  2.1
    1.0  3.0  1.5  2.9
    1.0  4.0  2.0  3.8
    1.0  5.0  2.5  4.5
]

y = [2.0, 3.9, 6.1, 8.0, 9.8]

β = X \ y
ŷ = X * β
residuals = y - ŷ
sse = sum(residuals .^ 2)

println("Linear regression")
println("β = ", β)
println("predictions = ", ŷ)
println("SSE = ", sse)

## 8) Designer-hours allocation problem

In [ ]:
scores = [
    90 80 10 50
    60 70 50 65
    70 40 80 85
]

required = [70, 50, 85, 35]
hours_available = 80

n_designers, n_projects = size(scores)

model_alloc = Model(HiGHS.Optimizer)
set_silent(model_alloc)

@variable(model_alloc, H[1:n_designers, 1:n_projects] >= 0)

@objective(model_alloc, Max, sum(scores[i, j] * H[i, j] for i in 1:n_designers, j in 1:n_projects))

@constraint(model_alloc, [i in 1:n_designers], sum(H[i, j] for j in 1:n_projects) <= hours_available)
@constraint(model_alloc, [j in 1:n_projects], sum(H[i, j] for i in 1:n_designers) >= required[j])

optimize!(model_alloc)

println("Designer allocation")
println("Maximum score = ", objective_value(model_alloc))
for i in 1:n_designers
    println("Designer ", i, ": ", value.(H[i, :]))
end

## 9) Diet problem

In [ ]:
cost = [1.0, 0.5, 2.0, 0.3]
cal = [100.0, 200.0, 150.0, 70.0]
protein = [0.5, 4.0, 8.0, 6.0]
vitamins = [2.0, 0.0, 10.0, 0.0]

model_diet = Model(HiGHS.Optimizer)
set_silent(model_diet)

@variable(model_diet, y[1:4] >= 0)

@objective(model_diet, Min, sum(cost[i] * y[i] for i in 1:4))
@constraint(model_diet, sum(cal[i] * y[i] for i in 1:4) >= 500)
@constraint(model_diet, sum(protein[i] * y[i] for i in 1:4) >= 50)
@constraint(model_diet, sum(vitamins[i] * y[i] for i in 1:4) >= 100)

optimize!(model_diet)

foods = ["Apple", "Bread", "Milk", "Egg"]

println("Diet problem")
println("Minimum cost = ", objective_value(model_diet))
for i in 1:4
    println(foods[i], ": ", value(y[i]))
end

## 10) Knapsack problem

In [ ]:
value_item = [120, 80, 60]
weight_item = [2.0, 1.0, 1.0]
capacity = 3.5

model_knapsack = Model(HiGHS.Optimizer)
set_silent(model_knapsack)

@variable(model_knapsack, x[1:3], Bin)

@objective(model_knapsack, Max, sum(value_item[i] * x[i] for i in 1:3))
@constraint(model_knapsack, sum(weight_item[i] * x[i] for i in 1:3) <= capacity)

optimize!(model_knapsack)

items = ["Tent", "Stove", "Food"]

println("Knapsack problem")
println("Maximum value = ", objective_value(model_knapsack))
for i in 1:3
    println(items[i], ": ", Int(round(value(x[i]))))
end